In [53]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
import numpy as np

In [30]:
train = pd.read_csv("/content/drive/MyDrive/dsn26/processed_train.csv")
test = pd.read_csv("/content/drive/MyDrive/dsn26/processed_test.csv")
y = pd.read_csv("/content/drive/MyDrive/dsn26/target.csv")

In [31]:
train.shape, test.shape, y.shape

((6818, 16), (1705, 16), (6818, 1))

In [32]:
n_train = train.shape[0]
n_train

6818

In [33]:
display(train.head())
print("\n\n")
display(test.head())

,id,product_weight_kg,fat_content,shelf_visibility,product_price,store_age_years,store_size,store_location_tier,store_format,price_vs_cat,price_rank_in_cat,price_vs_store,price_per_kg,weight_vs_cat,vis_vs_product,vis_x_price
0,row_00000,14.252000,20,0.0271,81.37,45,30,30,20,0.630238,0.234813,0.575641,5.709374,1.115616,0.409863,2.205127
1,row_00001,7.698000,20,0.0720,42.05,35,10,10,20,0.326007,0.086538,0.295775,5.462458,0.638414,1.088934,3.027600
2,row_00002,14.264000,10,0.0421,41.35,33,20,10,20,0.298018,0.023112,0.294545,2.898906,1.155650,0.636724,1.740835
3,row_00003,12.899667,10,0.0449,174.35,47,20,30,40,1.214306,0.732584,1.247467,13.515853,1.091619,0.679071,7.828315
4,row_00004,10.338000,10,0.0120,203.06,28,10,20,20,1.384185,0.868235,1.419477,19.642097,0.838375,0.181489,2.436720


,id,product_weight_kg,fat_content,shelf_visibility,product_price,store_age_years,store_size,store_location_tier,store_format,price_vs_cat,price_rank_in_cat,price_vs_store,price_per_kg,weight_vs_cat,vis_vs_product,vis_x_price
0,row_00009,8.628000,20,0.0167,190.72,23,20,30,30,1.244705,0.748352,1.345452,22.104775,0.643209,0.252572,3.185024
1,row_00015,6.993000,20,0.0579,261.07,28,10,20,20,2.024034,0.982692,1.824992,37.333047,0.579947,0.875684,15.115953
2,row_00019,17.167667,20,0.1262,112.50,47,20,30,40,0.782337,0.328328,0.804933,6.553016,1.307324,1.908659,14.197500
3,row_00020,14.034000,10,0.0761,200.71,30,10,20,20,1.554566,0.802570,1.424928,14.301696,1.098552,1.150943,15.274031
4,row_00023,13.063000,20,0.0650,150.59,33,20,10,20,1.166370,0.586449,1.072687,11.527980,1.022544,0.983065,9.788350


In [34]:
test_id = test['id']

In [35]:
train.drop("id", axis=1, inplace=True, errors = "ignore")
test.drop("id", axis=1, inplace=True, errors = "ignore")

In [38]:
categorical_cols = ["fat_content", "store_size", "store_location_tier", "store_format"]

for col in categorical_cols:
    if col in train.columns and train[col].dtype == 'int64':
        train[col] = train[col].astype('category')

for col in categorical_cols:
    if col in test.columns and test[col].dtype == 'int64':
        test[col] = test[col].astype('category')

In [39]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6818 entries, 0 to 6817
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   product_weight_kg    6818 non-null   float64 
 1   fat_content          6818 non-null   category
 2   shelf_visibility     6818 non-null   float64 
 3   product_price        6818 non-null   float64 
 4   store_age_years      6818 non-null   int64   
 5   store_size           6818 non-null   category
 6   store_location_tier  6818 non-null   category
 7   store_format         6818 non-null   category
 8   price_vs_cat         6818 non-null   float64 
 9   price_rank_in_cat    6818 non-null   float64 
 10  price_vs_store       6818 non-null   float64 
 11  price_per_kg         6818 non-null   float64 
 12  weight_vs_cat        6818 non-null   float64 
 13  vis_vs_product       6818 non-null   float64 
 14  vis_x_price          6818 non-null   float64 
dtypes: category(4), float

### Training base model

In [40]:
xgbr = XGBRegressor(eval_metric='rmse', objective = "reg:squarederror", enable_categorical = True)

In [41]:
xgbr.fit(train, y.total_sales)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [42]:
xgbr.score(train, y.total_sales)

0.9003849948429349

In [43]:
pd.DataFrame({"id": test_id, "total_sales": xgbr.predict(test)}).to_csv("submission.csv", index=False)

### Selecting Feature Importance

In [44]:
feature_importance = pd.DataFrame({
'Feature': train.columns,
'Importance': xgbr.feature_importances_ * 100
}).sort_values('Importance', ascending=False)

feature_importance

,Feature,Importance
7,store_format,40.845360
10,price_vs_store,26.564482
4,store_age_years,8.421947
3,product_price,5.197100
8,price_vs_cat,3.154043
9,price_rank_in_cat,2.456808
14,vis_x_price,2.181626
11,price_per_kg,2.047549
12,weight_vs_cat,1.949206
5,store_size,1.778999


In [45]:
train = train[["store_format", "price_vs_store", "store_age_years", "product_price"]]
test = test[["store_format", "price_vs_store", "store_age_years", "product_price"]]

In [47]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6818 entries, 0 to 6817
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   store_format     6818 non-null   category
 1   price_vs_store   6818 non-null   float64 
 2   store_age_years  6818 non-null   int64   
 3   product_price    6818 non-null   float64 
dtypes: category(1), float64(2), int64(1)
memory usage: 166.8 KB


In [48]:
xgbr = XGBRegressor(eval_metric='rmse', objective = "reg:squarederror", enable_categorical = True)

xgbr.fit(train, y.total_sales)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [49]:
xgbr.score(train, y.total_sales)

0.7291276781491992

### Hyperparameter Tuning

In [50]:
xgbr = XGBRegressor(eval_metric='rmse', objective = "reg:squarederror", enable_categorical = True)


In [51]:
param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}


In [54]:
grid_search = GridSearchCV(
    estimator=xgbr,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

In [55]:
print("Starting GridSearchCV...")
grid_search.fit(train, y.total_sales)
print("GridSearchCV completed.")

print(f"Best parameters found: {grid_search.best_params_}")
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best cross-validation RMSE: {best_rmse}")

Starting GridSearchCV...
Fitting 3 folds for each of 108 candidates, totalling 324 fits
GridSearchCV completed.
Best parameters found: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
Best cross-validation RMSE: 1081.8359800822616


In [57]:
best_xgbr = grid_search.best_estimator_
best_xgbr

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [58]:
xgbr = XGBRegressor(eval_metric='rmse', objective = "reg:squarederror", enable_categorical = True, **grid_search.best_params_)

In [59]:
xgbr.fit(train, y.total_sales)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [60]:
xgbr.score(train, y.total_sales)

0.6100875397151408

In [61]:
pd.DataFrame({"id": test_id, "total_sales": xgbr.predict(test)}).to_csv("submission.csv", index=False)